In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import re
import pandas as pd
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


# Loading data

In [11]:
data = pd.read_csv("bbc_news.csv")

In [12]:
data.head()

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?a...,With much of the UK enduring another period of...
1,1,9267,'Liz Truss the Brief?' World reacts to UK poli...,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_m...,The UK's political chaos has been watched arou...
2,2,7387,Rationing energy is nothing new for off-grid c...,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlan...,https://www.bbc.co.uk/news/uk-scotland-highlan...,Scoraig in the north west Highlands has long h...
3,3,767,The hunt for superyachts of sanctioned Russian...,"Tue, 22 Mar 2022 14:37:01 GMT",https://www.bbc.co.uk/news/60739336,https://www.bbc.co.uk/news/60739336?at_medium=...,"Wealthy Russians sanctioned by the US, EU and ..."
4,4,3712,Platinum Jubilee: 70 years of the Queen in 70 ...,"Wed, 01 Jun 2022 23:17:33 GMT",https://www.bbc.co.uk/news/uk-61660128,https://www.bbc.co.uk/news/uk-61660128?at_medi...,A quick look back at the Queen's 70 years on t...


In [13]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Unnamed: 0   1000 non-null   int64
 1   index        1000 non-null   int64
 2   title        1000 non-null   str  
 3   pubDate      1000 non-null   str  
 4   guid         1000 non-null   str  
 5   link         1000 non-null   str  
 6   description  1000 non-null   str  
dtypes: int64(2), str(5)
memory usage: 54.8 KB


In [14]:
titles = pd.DataFrame(data["title"])
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


# Cleaning the data

## Lowercasing the text

In [15]:
titles["title_lowercase"] = titles["title"].str.lower()
titles.head()

,title,title_lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...


## Removing Stopwords

In [17]:
en_stopwords = stopwords.words('english')
titles["title_with_no_stopwords"] = titles["title_lowercase"].apply(lambda x: " ".join([word for word in x.split() if word not in en_stopwords]))
titles.head()

,title,title_lowercase,title_with_no_stopwords
0,Can I refuse to work?,can i refuse to work?,refuse work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds


## Removing punctuations

In [18]:
titles["title_with_no_punc"] = titles["title_with_no_stopwords"].apply(lambda x: re.sub(r"[^\w\s]", "", x))
titles.head()

,title,title_lowercase,title_with_no_stopwords,title_with_no_punc
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds


## Tokenizing the text

In [20]:
titles["tokens_raw"] = titles["title"].apply(lambda x: word_tokenize(x))
titles["tokens_clean"] = titles["title_with_no_punc"].apply(lambda x: word_tokenize(x))

## Lemmatizing the Text

In [21]:
lm = WordNetLemmatizer()
titles["lemmatized_tokens_clean"] = titles["tokens_clean"].apply(lambda tokens: [lm.lemmatize(token) for token in tokens])
titles.head()

,title,title_lowercase,title_with_no_stopwords,title_with_no_punc,tokens_raw,tokens_clean,lemmatized_tokens_clean
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"[', Liz, Truss, the, Brief, ?, ', World, react...","[liz, truss, brief, world, reacts, uk, politic...","[liz, truss, brief, world, reacts, uk, politic..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[Rationing, energy, is, nothing, new, for, off...","[rationing, energy, nothing, new, offgrid, com...","[rationing, energy, nothing, new, offgrid, com..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[The, hunt, for, superyachts, of, sanctioned, ...","[hunt, superyachts, sanctioned, russian, oliga...","[hunt, superyachts, sanctioned, russian, oliga..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[Platinum, Jubilee, :, 70, years, of, the, Que...","[platinum, jubilee, 70, years, queen, 70, seco...","[platinum, jubilee, 70, year, queen, 70, second]"


In [25]:
tokens_raw_list = sum(titles["tokens_raw"], [])
tokens_clean_list = sum(titles["lemmatized_tokens_clean"], [])

# POS Tagging

In [26]:
nlp = spacy.load('en_core_web_sm')

In [28]:
spacy_doc = nlp(' '.join(tokens_raw_list))

In [29]:
pos_df = pd.DataFrame(columns=["token", "pos_tag"])

for token in spacy_doc:
    pos_df = pd.concat([pos_df,
                        pd.DataFrame.from_records([{"token":token.text, "pos_tag":token.pos_}])],
                       ignore_index=True)

In [31]:
pos_df_count = pos_df.groupby(["token", "pos_tag"]).size().reset_index(name="count").sort_values(by="count", ascending=False)

In [33]:
pos_df_count.head(10)

,token,pos_tag,count
93,:,PUNCT,543
6,',PUNCT,307
2893,in,ADP,187
4078,to,PART,176
3264,of,ADP,172
20,-,PUNCT,166
4039,the,DET,163
1852,and,CCONJ,147
13,'s,PART,143
95,?,PUNCT,130


In [34]:
nouns = pos_df_count[pos_df_count["pos_tag"] == "NOUN"][:10]
nouns

,token,pos_tag,count
4263,war,NOUN,35
3548,record,NOUN,15
4312,win,NOUN,15
4352,year,NOUN,14
3412,police,NOUN,14
4005,tax,NOUN,13
3057,living,NOUN,13
3364,people,NOUN,12
2322,day,NOUN,12
2565,fans,NOUN,11


In [35]:
verbs = pos_df_count[pos_df_count["pos_tag"] == "VERB"][:10]
verbs

,token,pos_tag,count
3683,says,VERB,30
2666,found,VERB,13
4313,win,VERB,11
4320,wins,VERB,10
2709,get,VERB,9
2384,dies,VERB,9
3104,make,VERB,8
3741,set,VERB,8
3986,take,VERB,8
3682,say,VERB,8


# NER

In [36]:
ner_df = pd.DataFrame(columns=["token", "ner_tag"])

for token in spacy_doc.ents:
    if pd.isna(token.label_) is False:
        ner_df = pd.concat([ner_df,
                            pd.DataFrame.from_records(
                                [{"token": token.text,
                                  "ner_tag": token.label_}])],
                           ignore_index=True)

In [37]:
ner_df.head()

,token,ner_tag
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP


In [38]:
ner_df_count = ner_df.groupby(["token", "ner_tag"]).size().reset_index(name="count").sort_values(by="count", ascending=False)

In [39]:
ner_df_count.head()

,token,ner_tag,count
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19
